##LOKAL OPEN SOURCE LLM - OLLAMA QWEN3:4B

In [1]:
!nvidia-smi

Sat Sep  5 20:32:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [3]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 118 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (402 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [4]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
!ollama --version

In [6]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [7]:
!cat /tmp/ollama.log

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAILvrj9gvlQIjUOKaMlfEMho3K23v5GPPhWXSvkWViHEu

time=2026-09-05T20:32:51.905Z level=INFO source=routes.go:1955 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost h

In [8]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [9]:
!ollama pull qwen3:4b

In [10]:
!ollama list

NAME        ID              SIZE      MODIFIED               
qwen3:4b    359d7dd4bcda    2.5 GB    Less than a second ago    


In [11]:
!pip -q install ollama

In [12]:
from ollama import chat

response = chat(
    model="qwen3:4b",
    messages=[
        {
            "role": "user",
            "content": "Explain AI agents in 3 simple sentences."
        }
    ]
)

print(response.message.content)

AI agents are intelligent systems that can perceive their environment, make decisions, and take actions to achieve goals.  
They sense data (like inputs or observations), reason through options, and then act to solve problems or complete tasks.  
Unlike passive tools, they work *independently* to handle real-world challenges without needing constant human intervention.


##ARAÇ/TOOL ÇAĞIRMA

In [13]:
#HESAPLAMA ARACI

In [14]:
def calculate(a, b, operation):

    if operation == "add":
        return a + b

    if operation == "subtract":
        return a - b

    if operation == "multiply":
        return a * b

    if operation == "divide":
        return a / b

    raise ValueError("Unknown operation")

In [15]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a basic calculation.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number"},
                    "b": {"type": "number"},
                    "operation": {
                        "type": "string",
                        "enum": [
                            "add",
                            "subtract",
                            "multiply",
                            "divide"
                        ]
                    }
                },
                "required": ["a", "b", "operation"]
            }
        }
    }
]

In [16]:
MODEL="qwen3:4b"

response = chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "1245 ve 37nin çarpımının sonucu nedir?"
        }
    ],
    tools=tools
)

response.message.tool_calls

[ToolCall(function=Function(name='calculate', arguments={'a': 1245, 'b': 37, 'operation': 'multiply'}))]

In [17]:
import json

messages = [
    {
        "role": "user",
        "content": "1245 ve 37nin çarpımının sonucu nedir?"
    }
]

for step in range(5):

    response = chat(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    if not response.message.tool_calls:
        print(response.message.content)
        break

    messages.append(response.message)

    for call in response.message.tool_calls:

        name = call.function.name
        args = call.function.arguments

        print("Tool:", name)
        print("Arguments:", args)

        result = calculate(**args)

        print("Result:", result)

        messages.append({
            "role": "tool",
            "tool_name": name,
            "content": json.dumps(result)
        })

Tool: calculate
Arguments: {'a': 1245, 'b': 37, 'operation': 'multiply'}
Result: 46065
The product of 1245 and 37 is **46065**.


In [18]:
MODEL="qwen3:4b"

response = chat(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Türkiyenin başkenti neresidir?"
        }
    ],
    tools=tools
)

response.message.tool_calls

In [19]:
response.message.tool_calls

In [20]:
import json

messages = [
    {
        "role": "user",
        "content": "Türkiyenin başkenti neresidir? Türkçe cevap ver"
    }
]

for step in range(5):

    response = chat(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    if not response.message.tool_calls:
        print(response.message.content)
        break

    messages.append(response.message)

    for call in response.message.tool_calls:

        name = call.function.name
        args = call.function.arguments

        print("Tool:", name)
        print("Arguments:", args)

        result = calculate(**args)

        print("Result:", result)

        messages.append({
            "role": "tool",
            "tool_name": name,
            "content": json.dumps(result)
        })

Türkiyenin başkenti Ankara'dır.


In [21]:
import json

messages = [
    {
        "role": "system",
        "content": "Sen Türkçe konuşan bir asistansın. Gelen soruları yanıtlamak için sadece sana sağlanan araçları (tools) kullan. Eğer soruyla alakalı bir araç sistemde tanımlı değilse, kendi iç bilgini kullanmak yerine kesinlikle 'Bununla alakalı bir tool sistemde yok, cevap veremiyorum.' de."
    },
    {
        "role": "user",
        "content": "Türkiye'nin başkenti neresidir?"
    }
]

for step in range(5):

    response = chat(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    if not response.message.tool_calls:
        print(response.message.content)
        break

    messages.append(response.message)

    for call in response.message.tool_calls:

        name = call.function.name
        args = call.function.arguments

        print("Tool:", name)
        print("Arguments:", args)

        result = calculate(**args)

        print("Result:", result)

        messages.append({
            "role": "tool",
            "tool_name": name,
            "content": json.dumps(result)
        })

Bununla alakalı bir tool sistemde yok, cevap veremiyorum.


In [ ]:
#ARAMA/SEARCH ARACI

In [22]:
knowledge_base = {

    "python": """
    Python, yapay zeka ve makine öğrenmesi alanında
    en yaygın kullanılan programlama dillerinden biridir.

    PyTorch, TensorFlow, scikit-learn ve birçok
    LLM kütüphanesi Python ekosisteminde bulunmaktadır.

    Özellikle veri bilimi, model geliştirme,
    prototipleme ve AI uygulamalarında güçlüdür.
    """,

    "java": """
    Java, kurumsal yazılım dünyasında yaygın kullanılan
    olgun bir programlama dilidir.

    Büyük ölçekli backend sistemlerinde,
    enterprise uygulamalarda ve dağıtık sistemlerde
    güçlü bir ekosisteme sahiptir.

    Üretim ortamlarında uzun yıllardır kullanılmaktadır.
    """,

    "go": """
    Go, sadelik, yüksek performans ve concurrency
    özellikleriyle öne çıkan bir programlama dilidir.

    Cloud-native uygulamalarda, backend sistemlerinde
    ve altyapı teknolojilerinde yaygın olarak kullanılır.
    """,

    "ai agent": """
    AI Agent, bir dil modelini araçlarla birleştirerek
    belirli görevleri yerine getiren bir sistemdir.

    Agent gerektiğinde bir araç seçebilir,
    aracın sonucunu gözlemleyebilir ve göreve
    devam edebilir.
    """
}

In [23]:
def search_knowledge(query: str) -> str:

    query = query.lower()

    results = []

    for keyword, information in knowledge_base.items():

        if keyword in query:
            results.append(information)

    if not results:
        return "Arama sonucunda ilgili bilgi bulunamadı."

    return "\n".join(results)

In [24]:
print(search_knowledge("Python yapay zeka için nasıl kullanılıyor?"))


    Python, yapay zeka ve makine öğrenmesi alanında
    en yaygın kullanılan programlama dillerinden biridir.

    PyTorch, TensorFlow, scikit-learn ve birçok
    LLM kütüphanesi Python ekosisteminde bulunmaktadır.

    Özellikle veri bilimi, model geliştirme,
    prototipleme ve AI uygulamalarında güçlüdür.
    


In [25]:
#RAPORLAMA ARACI

In [26]:
from pathlib import Path

def save_report(title: str, content: str):

    path = Path("agent_raporu.md")

    path.write_text(
        f"# {title}\n\n{content}",
        encoding="utf-8"
    )

    return f"Rapor başarıyla kaydedildi: {path}"

In [27]:
print(
    save_report(
        "Yapay Zeka Mühendisliği",
        "Python yapay zeka geliştirme açısından güçlü bir ekosisteme sahiptir."
    )
)

Rapor başarıyla kaydedildi: agent_raporu.md


In [28]:
#Tüm toolları tanımlıyoruz

tools = [

    {
        "type": "function",

        "function": {

            "name": "search_knowledge",

            "description": """
            Verilen konu hakkında bilgi araştırır.
            Bir konu hakkında bilgiye ihtiyaç olduğunda
            bu aracı kullan.
            """,

            "parameters": {

                "type": "object",

                "properties": {

                    "query": {
                        "type": "string",
                        "description": "Araştırılacak konu"
                    }

                },

                "required": ["query"]
            }
        }
    },

    {
        "type": "function",

        "function": {

            "name": "calculate",

            "description": """
            Matematiksel hesaplama yapar.
            Toplama, çıkarma, çarpma ve bölme işlemlerini
            gerçekleştirebilir.
            """,

            "parameters": {

                "type": "object",

                "properties": {

                    "a": {
                        "type": "number",
                        "description": "Birinci sayı"
                    },

                    "b": {
                        "type": "number",
                        "description": "İkinci sayı"
                    },

                    "operation": {
                        "type": "string",
                        "description": "Yapılacak matematiksel işlem",
                        "enum": [
                            "add",
                            "subtract",
                            "multiply",
                            "divide"
                        ]
                    }
                },

                "required": [
                    "a",
                    "b",
                    "operation"
                ]
            }
        }
    },

    {
        "type": "function",

        "function": {

            "name": "save_report",

            "description": """
            Hazırlanan sonucu Markdown formatında
            bir dosyaya kaydeder.
            """,

            "parameters": {

                "type": "object",

                "properties": {

                    "title": {
                        "type": "string",
                        "description": "Rapor başlığı"
                    },

                    "content": {
                        "type": "string",
                        "description": "Rapor içeriği"
                    }
                },

                "required": [
                    "title",
                    "content"
                ]
            }
        }
    }
]

In [29]:
tool_map = {
    "search_knowledge": search_knowledge,
    "calculate": calculate,
    "save_report": save_report
}

##FINAL AGENT

In [30]:
soru = """
Python, Java ve Go programlama dillerini
yapay zeka mühendisliği açısından karşılaştır.

Araştırma yap ve bir öneride bulun.

Sonucu Türkçe kısa bir rapor olarak hazırla
ve dosyaya kaydet.
"""

In [31]:
import json
from ollama import chat

MODEL = "qwen3:4b"


def run_agent(question):

    messages = [

        {
            "role": "system",
            "content": """
            Sen yardımcı bir Türkçe AI araştırma agentsın.

            Sana verilen görevi tamamlamak için
            gerektiğinde araçları kullan.

            Bir konu hakkında bilgi gerekiyorsa
            search_knowledge aracını kullan.

            Matematiksel hesaplama gerekiyorsa
            calculate aracını kullan.

            Kullanıcı rapor isterse
            save_report aracını kullan.

            Araçlardan gelen sonuçları kullanarak
            görevi tamamla.
            """
        },

        {
            "role": "user",
            "content": question
        }
    ]


    # Agent Loop
    for step in range(10):

        print(f"\n{'='*50}")
        print(f"🤖 AGENT ADIMI {step + 1}")
        print(f"{'='*50}")


        # LLM'e sor
        response = chat(
            model=MODEL,
            messages=messages,
            tools=tools
        )


        # Tool çağrısı yoksa → final cevap
        if not response.message.tool_calls:

            return response.message.content


        # Modelin mesajını conversation'a ekle
        messages.append(response.message)


        # Model hangi tool'u seçti?
        for call in response.message.tool_calls:

            tool_name = call.function.name
            arguments = call.function.arguments


            print(f"🔧 Seçilen tool: {tool_name}")
            print(f"📥 Parametreler: {arguments}")


            # Tool'u bul
            if tool_name not in tool_map:

                raise ValueError(
                    f"Bilinmeyen tool: {tool_name}"
                )


            # Gerçek Python fonksiyonunu çalıştır
            tool_function = tool_map[tool_name]

            result = tool_function(**arguments)


            print(f"📤 Tool sonucu: {result}")


            # Tool sonucunu tekrar LLM'e gönder
            messages.append({

                "role": "tool",

                "tool_name": tool_name,

                "content": json.dumps(
                    result,
                    ensure_ascii=False
                )
            })


    return "Agent maksimum adım sayısına ulaştı."


print("✅ run_agent() hazır!")

✅ run_agent() hazır!


In [32]:
cevap = run_agent(soru)

print("\n")
print("=" * 60)
print("🤖 AGENT'IN FİNAL CEVABI")
print("=" * 60)
print(cevap)


🤖 AGENT ADIMI 1
🔧 Seçilen tool: search_knowledge
📥 Parametreler: {'query': 'Python, Java ve Go programlama dilleri yapay zeka mühendisliği açısından karşılaştırması'}
📤 Tool sonucu: 
    Python, yapay zeka ve makine öğrenmesi alanında
    en yaygın kullanılan programlama dillerinden biridir.

    PyTorch, TensorFlow, scikit-learn ve birçok
    LLM kütüphanesi Python ekosisteminde bulunmaktadır.

    Özellikle veri bilimi, model geliştirme,
    prototipleme ve AI uygulamalarında güçlüdür.
    

    Java, kurumsal yazılım dünyasında yaygın kullanılan
    olgun bir programlama dilidir.

    Büyük ölçekli backend sistemlerinde,
    enterprise uygulamalarda ve dağıtık sistemlerde
    güçlü bir ekosisteme sahiptir.

    Üretim ortamlarında uzun yıllardır kullanılmaktadır.
    

    Go, sadelik, yüksek performans ve concurrency
    özellikleriyle öne çıkan bir programlama dilidir.

    Cloud-native uygulamalarda, backend sistemlerinde
    ve altyapı teknolojilerinde yaygın olarak kullanılır.

In [33]:
!ls

agent_raporu.md  sample_data
